## A ideia desse notebook é Extrair as informações das USGs e colocar em uma arquivo.parquet

**Biblioteca para mexer com o PDF e os pandas**

In [ ]:
!pip install PyPDF2
import PyPDF2
import pandas as pd
import os
import numpy as np





**Puxando as coisas do drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Extraindo pagina por página do PDF, encontrei esse modelo pronto na internet e parece que funcionou bem**


In [ ]:
def extrair_texto(input_pdf_path):
    with open(input_pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        textos_paginas = []
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            texto_pagina = page.extract_text()
            textos_paginas.append(texto_pagina)
    return textos_paginas

def processar_pasta(pasta_path):
    dados = []

    for arquivo in os.listdir(pasta_path):
        if arquivo.endswith('.pdf'):
            caminho_completo = os.path.join(pasta_path, arquivo)
            textos_paginas = extrair_texto(caminho_completo)
            for num_pagina, texto in enumerate(textos_paginas):
                dados.append({'Arquivo': arquivo, 'Página': num_pagina + 1, 'Texto': texto})

    df = pd.DataFrame(dados)
    return df

# Defina o caminho para a pasta que contém os PDFs
pasta_path = '/content/drive/MyDrive/PRIVATE_DATA/USG_Normal'
df = processar_pasta(pasta_path)

# Exibir o DataFrame
print(df)




In [ ]:
df

**Apenas pra visualização**

In [ ]:
conteudo_primeira_linha = df.iloc[5].Texto
for linha in conteudo_primeira_linha.split('\n'):
    print(linha)

##Capturando as Informações por Regex

In [ ]:
import re

def extrair_medidas(texto):
    padroes = {
        "nome_paciente": r'\d{2}/\d{2}/\d{4}\s+(.*?)\s*ULTRASSONOGRAFIA',
        "ID_paciente": r'(\d+)\s+\d{2}/\d{2}/\d{4}\s+(.*?)',
        "idade_gestacional_semanas": r'(\d+)\s+(semana|semanas)',
        "idade_gestacional_dias": r'(\d+)\s+(dia|dias)',
        "frequencia_cardiaca": r'(\d+)\s*(?:b\.p\.m\.|bpm)',
        "peso_fetal": r'(\d+)\s*(?:gramas)',
        "diametro_biparietal": r'DIÂMETRO BIPARIETAL:\s*(\d+[.,]?\d*)',
        "circunferência_cefálica": r'CIRCUNFERÊNCIA (?:CEFÁLICA|CRANIANA):\s*(\d+[.,]?\d*)',
        "circunferência_abdominal": r'CIRCUNFERÊNCIA ABDOMINAL:\s*(\d+[.,]?\d*)',
        "comprimento_femural": r'COMPRIMENTO FEMURAL:\s*(\d+[.,]?\d*)'
    }

    resultados = {}

    for medida, padrao in padroes.items():
        match = re.search(padrao, texto, re.IGNORECASE)
        if match:
            numero = match.group(1)
            if medida == "idade_gestacional_semanas":
                resultados[medida] = f"{numero} "
            elif medida == "idade_gestacional_dias":
                resultados[medida] = f"{numero} "
            else:
                resultados[medida] = f"{numero}mm"

    return resultados

############################################################3333
def extrair_data_nascimento(texto):
    padrao = r'US[^/]*?(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})'
    match = re.search(padrao, texto)

    if match:
        data_nascimento = match.group(1)
        return data_nascimento

    return None
###########################################################################

def extrair_data_exame(texto):
        # Encontrar a data no formato dd/mm/aaaa após uma sequência de 5 dígitos
    padrao = r'\d{5}.*?(\d{2}/\d{2}/\d{4})'

    match = re.search(padrao, texto)

    if match:
        return match.group(1)

    return None

########################################################


def formatar_idade_gestacional(row):
    semanas = row['Idade Gestacional Semanas']
    dias = row['Idade Gestacional Dias']
    return f'{semanas} semanas e {dias} dias'

#########################################################


dados_exames = []

for index, row in df.iterrows():
    texto = row['Texto']
    data_exame = extrair_data_exame(texto)
    data_nascimento = extrair_data_nascimento(texto)
    medidas = extrair_medidas(texto)

    # Adicionando os dados ao DataFrame
    dados_exames.append({
        "Paciente": medidas.get("nome_paciente", None),
        "Texto": texto,
        "ID_Paciente": medidas.get("ID_paciente", None),
        "Data do Exame": data_exame,
        "Data de Nascimento_Mãe": data_nascimento,
        "Idade Gestacional Semanas": medidas.get("idade_gestacional_semanas", None),
        "Idade Gestacional Dias": medidas.get("idade_gestacional_dias", None),
        "Frequencia Cardiaca": medidas.get("frequencia_cardiaca", None),
        "Peso Fetal": medidas.get('peso_fetal',None),
        "Diametro Biparietal": medidas.get('diametro_biparietal',None),
        "Circunferência Cefálica": medidas.get('circunferência_cefálica',None),
        "Circunferência Abdonominal": medidas.get('circunferência_abdominal',None),
        "Comprimento Femural": medidas.get('comprimento_femural',None)}),


# Criando o DataFrame
df_exames = pd.DataFrame(dados_exames)
df_exames['Paciente'] = df_exames['Paciente'].str.replace(r'\d+|mm|[^a-zA-Z\s]', '', regex=True)
df_exames['ID_Paciente'] = df_exames['ID_Paciente'].str.replace('mm$', '', regex=True)
df_exames['Frequencia Cardiaca'] = df_exames['Frequencia Cardiaca'].str.replace('mm$', 'bpm', regex=True)
df_exames['Peso Fetal'] = df_exames['Peso Fetal'].str.replace('mm$', 'g', regex=True)


df_exames

**A idade Gestacional esta como string por ter puxado do pdf, precisei transformar para número**

**A idade Gestacional Dias tinha alguns números maior que 7, o que não faz muito sentido, nesse caso estou atribuindo zero pois, na maioria das vezes esse número tá representando o retorno do paciente**

In [ ]:
# Converter as colunas 'Idade Gestacional Dias' e 'Idade Gestacional Semanas' para números
df_exames['Idade Gestacional Dias'] = pd.to_numeric(df_exames['Idade Gestacional Dias'], errors='coerce')
df_exames['Idade Gestacional Semanas'] = pd.to_numeric(df_exames['Idade Gestacional Semanas'], errors='coerce')

# Substituir valores maiores ou iguais a 7 por zero nas colunas 'Idade Gestacional Dias' e 'Idade Gestacional Semanas'
df_exames.loc[df_exames['Idade Gestacional Dias'] >= 7, 'Idade Gestacional Dias'] = 0

df_exames

**Alguns casos só tem a quantidade de semanas e não a quantidade de dias, o que iremos considerar que esse número é zero**

In [ ]:
# Verificar se 'Idade Gestacional Semanas' existe e 'Idade Gestacional Dias' está ausente
condicao = (df_exames['Idade Gestacional Semanas'].notna()) & (df_exames['Idade Gestacional Dias'].isna())

# Preencher 'Idade Gestacional Dias' com zero onde a condição for verdadeira
df_exames.loc[condicao, 'Idade Gestacional Dias'] = 0

df_exames

In [ ]:
df_exames = df_exames[df_exames['Paciente'].shift() != df_exames['Paciente']]
df_exames.reset_index(drop=True, inplace=True)

df_exames

In [ ]:
import sqlite3
conn = sqlite3.connect('USGs.db')

# Colocar o DataFrame em uma tabela chamada 'exames'
df_exames.to_sql('Normal', conn, if_exists='replace', index=False)

# Fechar a conexão
conn.close()

##EDA

In [ ]:
def remove_letters(value):
    return ''.join(filter(lambda x: x.isdigit() or x in ",.", value))

# Aplicar a função para as colunas especificadas
columns_to_clean = ['Frequencia Cardiaca','Peso Fetal', 'Diametro Biparietal', 'Circunferência Cefálica', 'Circunferência Abdonominal', 'Comprimento Femural']

for column in columns_to_clean:
    df_exames[column] = df_exames[column].astype(str).apply(remove_letters)

# Converter as colunas para o tipo numérico
for column in columns_to_clean:
    df_exames[column] = pd.to_numeric(df_exames[column].str.replace(',', '.'), errors='coerce')

df_exames

In [ ]:
import matplotlib.pyplot as plt
columns_to_plot = [
    'Idade Gestacional Semanas',
    'Frequencia Cardiaca',
    'Diametro Biparietal',
    'Circunferência Cefálica',
    'Circunferência Abdonominal',
    'Comprimento Femural'
]

# Criar gráficos de dispersão
for column in columns_to_plot:
    plt.figure(figsize=(10, 6))
    plt.scatter(df_exames.index, df_exames[column], alpha=0.7)
    none_count = df_exames[column].isna().sum()
    plt.title(f'Dispersão de {column} (None: {none_count})')
    plt.xlabel('Index')
    plt.ylabel(column)
    plt.grid(True)
    plt.show()

In [ ]:
for column in columns_to_plot:
    plt.figure(figsize=(10, 6))
    plt.boxplot(df_exames[column].dropna())
    plt.title(f'Boxplot de {column}')
    plt.ylabel(column)
    plt.grid(True)
    plt.show()

In [ ]:
df_exames.describe()

In [ ]:
def substituir_outliers_zscore(df, threshold=2):
    # Seleciona apenas as colunas numéricas
    cols_numericas = df.select_dtypes(include=['float64', 'int64']).columns

    for col in cols_numericas:
        # Calcula o z-score para a coluna
        z_scores = (df[col] - df[col].mean()) / df[col].std()

        # Substitui os outliers por None
        df.loc[np.abs(z_scores) > threshold, col] = None

    return df


df_exames_sem_outliers = substituir_outliers_zscore(df_exames, threshold=2)
print(df_exames_sem_outliers)

In [ ]:
import matplotlib.pyplot as plt
columns_to_plot = [
    'Idade Gestacional Semanas',
    'Frequencia Cardiaca',
    'Diametro Biparietal',
    'Circunferência Cefálica',
    'Circunferência Abdonominal',
    'Comprimento Femural'
]

# Criar gráficos de dispersão
for column in columns_to_plot:
    plt.figure(figsize=(10, 6))
    plt.scatter(df_exames_sem_outliers.index, df_exames_sem_outliers[column], alpha=0.7)
    none_count = df_exames_sem_outliers[column].isna().sum()
    plt.title(f'Dispersão de {column} (None: {none_count})')
    plt.xlabel('Index')
    plt.ylabel(column)
    plt.grid(True)
    plt.show()

In [ ]:
df_exames_sem_outliers.describe()